# VietMed correction: end-to-end LoRA training

Notebook thực thi R0/R1/R2 và R3 seeds 42/43/44 theo `RESEARCH_PLAN.md`; xuất model độc lập. Full mode không giới hạn mẫu. Nguồn phụ tắt khi chưa có phê duyệt; không tuyên bố an toàn y khoa.

## Khởi động từ repo root
```bash
uv python install 3.12
uv venv --python 3.12 experiments/002-vietmed-correction-training/.venv
uv pip sync --python experiments/002-vietmed-correction-training/.venv/bin/python experiments/002-vietmed-correction-training/requirements.lock --extra-index-url https://download.pytorch.org/whl/cu128
experiments/002-vietmed-correction-training/.venv/bin/python experiments/002-vietmed-correction-training/scripts/execute_notebook.py --mode full
```

Hoặc mở notebook với kernel từ môi trường trên và **Restart Kernel → Run All**. `VODOCO_PROJECT_ROOT` cấu hình root nếu notebook server chạy ngoài repo; `CORRECTION_MODE=smoke` dùng namespace riêng để gỡ lỗi, không được coi là full training.

## 1. Cấu hình và preflight
Paths được resolve từ repo, không hardcode máy cá nhân. Kiểm tra version trước khi import stack training.

In [1]:
import os, sys, importlib.metadata
from pathlib import Path

MODE = os.environ.get('CORRECTION_MODE', 'full')
if os.environ.get('VODOCO_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['VODOCO_PROJECT_ROOT']).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'RESEARCH_PLAN.md').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Set VODOCO_PROJECT_ROOT before launching this notebook outside the repo')
EXPERIMENT = PROJECT_ROOT / 'experiments/002-vietmed-correction-training'
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select the pinned Python 3.12 environment described above')
for line in (EXPERIMENT / 'requirements.lock').read_text().splitlines():
    if not line or line.startswith('#') or '==' not in line:
        continue
    package, expected = line.split('==', 1)
    if importlib.metadata.version(package).lower() != expected.lower():
        raise RuntimeError(f'Environment mismatch: {package}; sync requirements.lock before starting kernel')
sys.path.insert(0, str(EXPERIMENT / 'scripts'))
from runtime import make_config, inspect_runtime, prepare_models, lock_recipe
from common import read_json, release_gpu
from data_pipeline import prepare_data, generate_asr_cache, build_pairs
from pipeline import (tuning_records, baselines, train_all, selected_dev_results, lock_final_selection, final_evaluation, export_final_model, final_summary)
from training import training_smoke
import pandas as pd
from transformers import AutoTokenizer
cfg = make_config(PROJECT_ROOT, MODE)
display({'mode': MODE, 'project_root': str(PROJECT_ROOT), 'training': cfg['training'], 'generation': cfg['generation'], 'seeds': cfg['seeds']})

/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'mode': 'full',
 'project_root': '/home/hieu/Work/Learning.UIT/VoDoCo',
 'training': {'rank': 16,
  'alpha': 32,
  'dropout': 0.05,
  'learning_rate': 0.0001,
  'weight_decay': 0.01,
  'max_grad_norm': 1.0,
  'warmup_ratio': 0.05,
  'batch_size': 2,
  'gradient_accumulation_steps': 8,
  'epochs': 5,
  'patience': 2,
  'errorful_fraction': 0.7},
 'generation': {'num_beams': 4, 'do_sample': False, 'max_new_tokens': 512},
 'seeds': [42, 43, 44]}

## 2. Môi trường, GPU và reproducibility
Đo CUDA GEMM + autograd thật. Pin libraries, precision, source hashes và code provenance.

In [2]:
environment = inspect_runtime(cfg)
display({k: environment[k] for k in ['python', 'libraries', 'gpu', 'gpu_total_bytes', 'compute_capability', 'cuda', 'precision', 'cuda_forward_backward_loss']})

{'python': '3.12.13',
 'libraries': {'torch': '2.8.0+cu128',
  'transformers': '4.57.6',
  'peft': '0.17.1',
  'accelerate': '1.10.1',
  'datasets': '4.1.1',
  'pyarrow': '21.0.0',
  'soundfile': '0.13.1',
  'jiwer': '4.0.0',
  'sentencepiece': '0.2.1',
  'numpy': '2.2.6',
  'scipy': '1.16.2',
  'librosa': '0.11.0',
  'huggingface-hub': '0.36.2',
  'nbclient': '0.10.2',
  'nbformat': '5.10.4',
  'ipykernel': '6.30.1'},
 'gpu': 'NVIDIA GeForce RTX 5060 Ti',
 'gpu_total_bytes': 16660430848,
 'compute_capability': [12, 0],
 'cuda': '12.8',
 'precision': 'bf16',
 'cuda_forward_backward_loss': 125.97351837158203}

## 3. Nguồn dữ liệu và split audit
Verify 16 file VietMed, decode đủ 9.207 audio và kiểm tra PCM duplicates. Giữ official test; tách recording `VietMed_019` khỏi tuning. Audio/raw reference không sửa.

In [3]:
records_by_split = prepare_data(cfg)
split_audit = read_json(Path(cfg['derived_dir']) / MODE / 'manifests/splits.json')
display(pd.DataFrame([{'split': split, 'selected_rows': len(rows), 'train_duplicate_exclusions': sum(r['duplicate_excluded'] for r in rows)} for split, rows in records_by_split.items()]))
display({'derived_counts': split_audit['derived_counts'], 'train_excluded_ids': split_audit['train_excluded_ids'], 'full_audio_audit_count': split_audit['full_audio_audit_count']})

,split,selected_rows,train_duplicate_exclusions
0,train,2773,0
1,dev,2912,0
2,test,3437,0
3,cv,85,0


{'derived_counts': {'train_real': 2773,
  'dev_tune': 2769,
  'dev_shared_recording': 143,
  'test_official': 3437,
  'cv_diagnostic': 85},
 'train_excluded_ids': [],
 'full_audio_audit_count': 9207}

## 4. Model/processor snapshots
Model/processor SHA được khóa; các file local được đối chiếu hash upstream. Không trộn PhoWhisper với MultiMed-ST.

In [4]:
models = prepare_models(cfg)
display(pd.DataFrame([{'role': kind, 'repo': value['repo_id'], 'revision': value['revision'], 'verified_files': len(value['hashes'])} for kind, value in models.items()]))

Verified asr: leduckhai/MultiMed-ST@fb15edd1dfc68810a7d0c75e6cfc73c3e5ed01c1 (11 files)


Verified correction: bmd1905/vietnamese-correction-v2@a3d342348d39d87622aa58aabd07de38ade8c17b (6 files)


Verified ner: leduckhai/VietMed-NER@cccffb7de14423114f7d4bafc9f736b9d866e446 (6 files)


,role,repo,revision,verified_files
0,asr,leduckhai/MultiMed-ST,fb15edd1dfc68810a7d0c75e6cfc73c3e5ed01c1,11
1,correction,bmd1905/vietnamese-correction-v2,a3d342348d39d87622aa58aabd07de38ade8c17b,6
2,ner,leduckhai/VietMed-NER,cccffb7de14423114f7d4bafc9f736b9d866e446,6


## 5. Cache ASR train/dev
Frozen MultiMed-ST, 16 kHz, deterministic decoding. Audio dài dùng native long-form; không crop reference. Resume chỉ khi signature, ID, reference và audio hash khớp.

In [5]:
train_records = generate_asr_cache(cfg, records_by_split['train'], 'train')
dev_all = generate_asr_cache(cfg, records_by_split['dev'], 'dev')
dev_records = tuning_records(dev_all)
release_gpu()
display({'train': len(train_records), 'dev_all': len(dev_all), 'dev_tune': len(dev_records), 'empty_train_hypotheses': sum(r['hypothesis_empty'] for r in train_records)})

`torch_dtype` is deprecated! Use `dtype` instead!


{'train': 2773, 'dev_all': 2912, 'dev_tune': 2769, 'empty_train_hypotheses': 0}

## 6. D_real và identity supervision
Hypothesis→reference chỉ từ train. Giữ errorful và identity; audit parent IDs và token lengths. Nếu dài, chia cửa sổ input/target theo alignment, không truncate độc lập.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(cfg['models']['correction']['path'], local_files_only=True)
pairs = build_pairs(cfg, train_records, tokenizer)
pair_audit = read_json(Path(cfg['derived_dir']) / MODE / 'pairs/manifest.json')
display({k: pair_audit[k] for k in ['eligible_parent_count', 'pair_count', 'pair_type_counts', 'errorful_pairs', 'real_errorful_parents', 'identity_unique_views', 'windowed_views', 'source_token_max', 'target_token_max']})

{'eligible_parent_count': 2773,
 'pair_count': 5544,
 'pair_type_counts': {'real': 2773, 'identity': 2771},
 'errorful_pairs': 541,
 'real_errorful_parents': 541,
 'identity_unique_views': 2771,
 'windowed_views': 0,
 'source_token_max': 44,
 'target_token_max': 44}

## 7. Baseline và evaluation
R0 không sửa; R1 train-only rules có provenance; R2 zero-shot. Khóa review sample trước corrector outputs. Corpus WER giữ insertion của empty reference; preservation là probe, không phải clinical safety.

In [7]:
baseline_result = baselines(cfg, train_records, dev_records)
display(pd.DataFrame([{'run': run, 'wer': report['summary']['wer'], 'cer': report['summary']['cer'], 'overcorrection': report['summary']['reference_overcorrection_rate']} for run, report in baseline_result['reports'].items()]))

{
  "R0": {
    "wer": 0.04436209473266844,
    "preservation": 0.0
  },
  "R1": {
    "wer": 0.04436209473266844,
    "preservation": 0.0
  },
  "R2": {
    "wer": 0.05247506633726782,
    "preservation": 0.19934994582881907
  }
}


,run,wer,cer,overcorrection
0,R0,0.044362,0.035780,0.00000
1,R1,0.044362,0.035780,0.00000
2,R2,0.052475,0.042811,0.19935


## 8. Training smoke: gradients, cập nhật, save/load
Hai optimizer updates trên train thật, tách khỏi các lượt chính. Chứng minh base frozen, adapter thay đổi, padding và loss đúng, deterministic reload và peak VRAM.

In [8]:
smoke_evidence = training_smoke(cfg, pairs)
display(smoke_evidence)

`torch_dtype` is deprecated! Use `dtype` instead!


{'losses': [1.8327947854995728, 1.9543851613998413],
 'gradient_norms': [0.6885833144187927, 0.6924562454223633],
 'base_unchanged': True,
 'adapter_updated': True,
 'trainable_parameters': 2359296,
 'peak_vram_bytes': 1919235584,
 'seconds': 1.5961980290012434,
 'smoke_pair_ids': ['real:f343e5c71f609b7cec5e71b71b461f8dc73e44196198a22c9b49721fda8bfa3a:0000',
  'real:cdf6d49d83f9e1110d9a2d3b071f9772f8376de9cbd62d6ccb7e22299dcd4c61:0000',
  'real:434f7a48b2e32e9f598dce7c87cbbf993167ba14ae6b5c8bcfc4add265313940:0000',
  'real:f75f9b1593437e4e7de6bc6938681be77cb8d1189ecd9c1c09e4e2bba8f7e67f:0000',
  'real:24ebf5cc09130eb9e0b5566536135f79d3d9417f1c22bd0e14710f7cf7056c31:0000',
  'real:505f72b99c909d4b21745a7e5ce16b1a2541a381ce6f816f34223e3990ae5daf:0000',
  'real:53df11ca5425ae34e92a340563a2ac716750572a10a450337824f3e0ee904383:0000',
  'real:fb8a78d74f2d42f6dff3f329526fd4e201b0b5b933cd50b9c75ec512c1289690:0000'],
 'save_reload_outputs_equal': True,
 'adapter_hashes': {'adapter_config.json':

## 9. Khóa recipe và full LoRA training
Không tune bằng test. Seeds 42/43/44 khởi tạo độc lập, sampler 70/30, token-weighted accumulation, dev generation mỗi epoch. Full tối đa 5 epoch, early stopping patience 2.

In [9]:
recipe_lock = lock_recipe(cfg, pairs, dev_records)
display({'recipe_signature': recipe_lock['signature'], 'locked_at': recipe_lock['locked_at']})
training_receipts = train_all(cfg, pairs, dev_records)
display(pd.DataFrame([{'run': r['run_name'], 'seed': r['seed'], 'epochs': r['completed_epochs'], 'optimizer_steps': r['actual_optimizer_steps'], 'best_checkpoint': r['best_checkpoint'], 'dev_wer': r['best_key'][0], 'peak_vram_GiB': r['peak_vram_bytes'] / 2**30} for r in training_receipts]))

{'recipe_signature': 'f47425309d20ccf15466a080a800989391b0388b567a0ba2eb7e9eec4d4741cd',
 'locked_at': '2026-09-06T10:23:55.224289+00:00'}

Verified completed R3_seed42: 1041 updates


Verified completed R3_seed43: 1041 updates


Verified completed R3_seed44: 1041 updates


,run,seed,epochs,optimizer_steps,best_checkpoint,dev_wer,peak_vram_GiB
0,R3_seed42,42,3,1041,epoch-1,0.053619,1.953755
1,R3_seed43,43,3,1041,epoch-1,0.054183,1.951381
2,R3_seed44,44,3,1041,epoch-1,0.057340,1.951381


## 10. Chọn checkpoint, NER consistency và test lock
Chọn trained candidate trên dev, không giả rằng đã review y khoa. Frozen NER chạy trên raw/corrected/reference; chỉ báo prediction consistency. Xuất blinded review và paired cluster bootstrap.

In [10]:
dev_result = selected_dev_results(cfg, dev_records, training_receipts, baseline_result)
selection = lock_final_selection(cfg, training_receipts, dev_result)
display(selection)
display({'review': dev_result['review'], 'ner_consistency': dev_result['ner']['runs'], 'auxiliary': cfg['auxiliary']})

{'selected_run': 'R3_seed42',
 'seed': 42,
 'checkpoint': 'epoch-1',
 'checkpoint_hashes': {'adapter_config.json': '3547ccd10f716824b434b28ca4024039fb353831b6734c2146f39105070a025d',
  'adapter_model.safetensors': '8a6c9ff21c125c2498e3fac0f0bde9225031cf780940ac79569a5694567cb6c7'},
 'recipe_signature': 'f47425309d20ccf15466a080a800989391b0388b567a0ba2eb7e9eec4d4741cd',
 'mode': 'full',
 'all_runs': [{'run_name': 'R3_seed42',
   'best_checkpoint': 'epoch-1',
   'best_hashes': {'adapter_config.json': '3547ccd10f716824b434b28ca4024039fb353831b6734c2146f39105070a025d',
    'adapter_model.safetensors': '8a6c9ff21c125c2498e3fac0f0bde9225031cf780940ac79569a5694567cb6c7'},
   'dev_key': [0.05361881233415683, 0.2195738533766703, 1]},
  {'run_name': 'R3_seed43',
   'best_checkpoint': 'epoch-1',
   'best_hashes': {'adapter_config.json': '3547ccd10f716824b434b28ca4024039fb353831b6734c2146f39105070a025d',
    'adapter_model.safetensors': '51e4f282a529dbf3e19716ea003d61fed4f50b407d28409a44ee61f32fe0

{'review': {'assignment': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/outputs/full/review/sample_assignment.json',
  'blinded_csv': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/outputs/full/review/blinded_review.csv',
  'blinding_key': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/outputs/full/review/blinding_key.json',
  'export_state': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/outputs/full/review/export_state.json'},
 'ner_consistency': {'R0': {'num_samples': 2769,
   'matched': 7634,
   'reference_predicted_entities': 8179,
   'candidate_predicted_entities': 8097,
   'consistency_precision': 0.9428183277757194,
   'consistency_retention': 0.9333659371561316,
   'consistency_f1': 0.9380683214549029,
   'both_entity_sets_empty_samples': 224,
   'interpretation': 'NER prediction consistency, not gold NER F1 or clinical quality',
   'zero_den

## 11. Final evaluation và diagnostic sau test lock
Chấm toàn official test, giữ riêng 143 dev_shared_recording và 85 cv_diagnostic. Không dùng diagnostic để chọn model. Báo mọi seed/run, mean/std và 2.000 paired recording-cluster bootstrap. Test đã được xem trong lịch sử.

In [11]:
test_result = final_evaluation(cfg, records_by_split, training_receipts, baseline_result['rules'])
display(pd.DataFrame([{'run': run, 'samples': report['summary']['num_samples'], 'wer': report['summary']['wer'], 'cer': report['summary']['cer'], 'relative_wer_reduction': report['summary']['relative_wer_reduction'], 'overcorrection': report['summary']['reference_overcorrection_rate']} for run, report in test_result['reports'].items()]))
display(test_result['statistics'])

,run,samples,wer,cer,relative_wer_reduction,overcorrection
0,R0,3437,0.283485,0.213638,0.000000,0.000000
1,R1,3437,0.283485,0.213638,0.000000,0.000000
2,R2,3437,0.278954,0.215046,0.015984,0.190864
3,R3_seed42,3437,0.279834,0.216006,0.012880,0.173407
4,R3_seed43,3437,0.278613,0.215186,0.017189,0.179226
5,R3_seed44,3437,0.278980,0.216417,0.015891,0.215595


{'split_name': 'test_official',
 'runs': {'R0': {'wer': 0.2834852504662586,
   'raw_wer': 0.2834852504662586,
   'delta_wer_to_R0': 0.0,
   'ci95_percentile': [0.0, 0.0],
   'defined_replicates': 2000,
   'undefined_replicates': 0,
   'cluster_errors': {'VietMed_002': 3276,
    'VietMed_004': 3781,
    'VietMed_014': 1260,
    'VietMed_015': 2274,
    'VietMed_017': 838,
    'VietMed_018': 3128,
    'VietMed_019': 979,
    'VietMed_023': 1454,
    'VietMed_024': 1459,
    'VietMed_025': 1033,
    'VietMed_026': 150,
    'VietMed_027': 605,
    'VietMed_028': 790,
    'VietMed_029': 557}},
  'R1': {'wer': 0.2834852504662586,
   'raw_wer': 0.2834852504662586,
   'delta_wer_to_R0': 0.0,
   'ci95_percentile': [0.0, 0.0],
   'defined_replicates': 2000,
   'undefined_replicates': 0,
   'cluster_errors': {'VietMed_002': 3276,
    'VietMed_004': 3781,
    'VietMed_014': 1260,
    'VietMed_015': 2274,
    'VietMed_017': 838,
    'VietMed_018': 3128,
    'VietMed_019': 979,
    'VietMed_023': 14

## 12. Export, FP32 control, reload và chấm file đã giao
Kiểm tra đại số merge ở FP32 và đo sai khác BF16, không giả định hai nhánh làm tròn giống nhau. Giữ nguyên checkpoint/precision/decoder đã khóa. Merged safetensors được reload local-only, chấm lại toàn official test và NER consistency; không dùng kết quả này để chọn lại model. Demo chỉ nhận ASR text, không nhận reference.

In [12]:
export_result = export_final_model(cfg, selection, train_records)
export_evidence = read_json(Path(cfg['outputs_dir']) / 'export-verification.json')
display({'model_directory': export_result['model_directory'], 'verification': export_result['verification'], 'exported_test': {k: export_result['exported_test_summary'][k] for k in ['num_samples', 'wer', 'cer', 'reference_overcorrection_rate']}})
display(pd.DataFrame(export_evidence['demo']))

{'model_directory': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/models/final_model',
 'verification': {'adapter_vs_merged_equal': False,
  'differing_probe_ids': ['train:utt_id_train_000008'],
  'merged_vs_reloaded_equal': True,
  'probe_count': 8,
  'precision_control': {'fp32_outputs_equal': True,
   'max_abs_logit_difference': 0.000823974609375,
   'mean_abs_logit_difference': 4.0547645767219365e-05,
   'atol': 0.001,
   'rtol': 0.0001,
   'explanation': 'BF16 separately rounds base and LoRA branches; merged weights round after addition.',
   'fp32_is_diagnostic_only': True,
   'deployment_precision_unchanged': 'bf16'},
  'inference_local_files_only': True,
  'reference_passed_to_inference': False},
 'exported_test': {'num_samples': 3437,
  'wer': 0.2792429535842812,
  'cer': 0.21555587323094516,
  'reference_overcorrection_rate': 0.17951702065755018}}

,id,asr_input,corrected
0,train:utt_id_train_000001,cái gì có phải là do cái cơn khó thở hay là cò...,cái gì có phải là do cái cơn khó thở hay là cò...
1,train:utt_id_train_000002,phải mang nịt rồi phải đi châm cứu rồi cho uốn...,phải mang nịt rồi phải đi châm cứu rồi cho uốn...
2,train:utt_id_train_000003,chưa mạch máu của chúng ta vấn đề gì chưa tim ...,chưa mạch máu của chúng ta vấn đề gì chưa tim ...
3,train:utt_id_train_000004,cho tất cả những khán giả mắc những vấn đề liê...,cho tất cả những khán giả mắc những vấn đề liê...
4,train:utt_id_train_000005,bài tập nặng mạnh để hi vọng rằng nó vượt qua ...,bài tập nặng mạnh để hi vọng rằng nó vượt qua ...
5,train:utt_id_train_000006,khớp thì chúng ta thường nói tới đối tượng là ...,khớp thì chúng ta thường nói tới đối tượng là ...
6,train:utt_id_train_000007,giận nó hay đau đau cả cả ngày đêm nhưng,giận nó hay đau đau cả cả ngày đêm nhưng
7,train:utt_id_train_000008,khớp tại việt nam để sản xuất nên cái thực phẩ...,khớp tại Việt nam để sản xuất nên cái thực phẩ...


## 13. Tổng hợp, artifacts và hạn chế

- Full R3 gồm seeds 42/43/44, chọn epoch bằng generated dev WER; test không dùng để chọn seed. Early stopping là hoàn tất đúng protocol, không phải lượt bị ngắt.
- R1 không có rule đạt các ngưỡng train-only đã chốt nên output trùng R0. Không nới ngưỡng sau khi chấm test.
- Bảng R3 đo adapter; file merged có thể khác do BF16, được đo riêng trong section 12 và `outputs/full/exported-model-test.json`. Không thay thế số liệu adapter bằng số của file xuất.
- Overcorrection là tỷ lệ reference bị đổi sau scoring normalization; NER consistency không phải gold NER F1. Review con người còn pending; không có kết luận an toàn y khoa hoặc tiết kiệm công chỉnh sửa.
- Test đã được tiếp xúc trong thí nghiệm lịch sử. Chỉ có ít recording; CI cluster-bootstrap không bao phủ mọi nguồn bất định. R4/R3-budget tắt do thiếu approved auxiliary manifest.

### Artifact và chạy lại

Trong thư mục experiment: `models/full/R3_seed{42,43,44}/epoch-1/` giữ adapters đã chọn; `models/final_model/` là Transformers directory độc lập. `outputs/full/final-summary.json`, `results.csv`, `bootstrap/`, `evaluation/`, `ner/`, `review/` và `export-verification.json` giữ kết quả và provenance. Cache/split/pairs ở `datasets/derived/correction/full/` tính từ repo root.

Lệnh full ở đầu notebook kiểm tra hashes, signatures, identities và completed receipts trước khi reuse. `--mode smoke` ghi namespace riêng; không thay notebook full đã chạy. Khi chuyển máy, cần môi trường pinned, source snapshots theo `datasets/sources.json` và các checkpoint/cache tương ứng; root được cấu hình qua `VODOCO_PROJECT_ROOT`. Artifact hiện tại ghi absolute paths: có thể cần tạo namespace/cache mới khi chuyển máy, không coi việc copy folder là đã xác minh resume. Không sửa recipe-lock/test-lock để ép reuse một protocol khác. Models, datasets và output chứa text/review không tự commit/push.


In [13]:
summary = final_summary(cfg, training_receipts, dev_result, test_result, export_result)
display(pd.DataFrame(summary['metrics']))
display({'notebook': summary['notebook'], 'final_model': export_result['model_directory'], 'outputs': cfg['outputs_dir'], 'full_training_completed': summary['complete_full_training'], 'human_review': summary['human_review'], 'auxiliary': summary['auxiliary']})
if MODE == 'full':
    assert summary['complete_full_training']
    assert all(report['summary']['num_samples'] == 3437 for report in test_result['reports'].values())
assert export_result['verification']['merged_vs_reloaded_equal']
if MODE == 'full':
    display({split: read_json(Path(cfg['outputs_dir']) / 'bootstrap' / split / 'comparison.json')['R3_seeds'] for split in ['dev_tune', 'test_official']})


[
  {
    "split": "dev_tune",
    "run": "R0",
    "samples": 2769,
    "wer": 0.04436209473266844,
    "cer": 0.03577951575890975,
    "relative_wer_reduction": 0.0,
    "reference_overcorrection_rate": 0.0,
    "improved": 0,
    "worsened": 0,
    "ner_consistency_f1": 0.9380683214549029
  },
  {
    "split": "dev_tune",
    "run": "R1",
    "samples": 2769,
    "wer": 0.04436209473266844,
    "cer": 0.03577951575890975,
    "relative_wer_reduction": 0.0,
    "reference_overcorrection_rate": 0.0,
    "improved": 0,
    "worsened": 0,
    "ner_consistency_f1": 0.9380683214549029
  },
  {
    "split": "dev_tune",
    "run": "R2",
    "samples": 2769,
    "wer": 0.05247506633726782,
    "cer": 0.042811494691792545,
    "relative_wer_reduction": -0.18288071502234446,
    "reference_overcorrection_rate": 0.19934994582881907,
    "improved": 61,
    "worsened": 512,
    "ner_consistency_f1": 0.8171806167400881
  },
  {
    "split": "dev_tune",
    "run": "R3_seed42",
    "samples": 2769,

,split,run,samples,wer,cer,relative_wer_reduction,reference_overcorrection_rate,improved,worsened,ner_consistency_f1
0,dev_tune,R0,2769,0.044362,0.035780,0.000000,0.000000,0,0,0.938068
1,dev_tune,R1,2769,0.044362,0.035780,0.000000,0.000000,0,0,0.938068
2,dev_tune,R2,2769,0.052475,0.042811,-0.182881,0.199350,61,512,0.817181
3,dev_tune,R3_seed42,2769,0.053619,0.045005,-0.208663,0.219574,71,552,0.913387
4,dev_tune,R3_seed43,2769,0.054183,0.045409,-0.221382,0.225713,70,571,0.914688
5,dev_tune,R3_seed44,2769,0.057340,0.048745,-0.292540,0.284940,84,720,0.908676
6,test_official,R0,3437,0.283485,0.213638,0.000000,0.000000,0,0,0.589956
7,test_official,R1,3437,0.283485,0.213638,0.000000,0.000000,0,0,0.589956
8,test_official,R2,3437,0.278954,0.215046,0.015984,0.190864,583,298,0.587081
9,test_official,R3_seed42,3437,0.279834,0.216006,0.012880,0.173407,553,315,0.604422


{'notebook': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/train_correction_end_to_end.ipynb',
 'final_model': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/models/final_model',
 'outputs': '/home/hieu/Work/Learning.UIT/VoDoCo/experiments/002-vietmed-correction-training/outputs/full',
 'full_training_completed': True,
 'human_review': 'pending; no clinical safety claim',
 'auxiliary': {'enabled': False,
  'runs_omitted': ['R4', 'R3-budget'],
  'reason': 'No approved rights/text-review/source-group/noise manifest; R3 is independent.',
  'required_before_enable': ['rights',
   'reviewed clean targets',
   'parent/source groups',
   'train-only noise profile',
   'matched-budget R3-budget']}}

{'dev_tune': {'runs_by_seed': {'42': 'R3_seed42',
   '43': 'R3_seed43',
   '44': 'R3_seed44'},
  'missing_protocol_seeds': [],
  'status': 'complete',
  'wer_mean': 0.055047224001382664,
  'wer_std': 0.0020053728132552384,
  'delta_wer_mean': 0.010685129268714226,
  'delta_wer_std': 0.002005372813255238,
  'std_policy': 'sample standard deviation across seeds, ddof=1; null for fewer than two defined seeds',
  'mean_delta_ci95_percentile': [0.008927211099268865, 0.011889405365500209],
  'mean_delta_defined_replicates': 2000,
  'resampling_policy': 'same recording draws for every seed, mean of per-seed ratios; never pooled utterances'},
 'test_official': {'runs_by_seed': {'42': 'R3_seed42',
   '43': 'R3_seed43',
   '44': 'R3_seed44'},
  'missing_protocol_seeds': [],
  'status': 'complete',
  'wer_mean': 0.2791422592310454,
  'wer_std': 0.0006266376205662309,
  'delta_wer_mean': -0.004342991235213253,
  'delta_wer_std': 0.0006266376205662374,
  'std_policy': 'sample standard deviation acr